# Custom surface site analysis

Edit **Cell 2 only**, then **Run All**.


In [21]:
# Imports (DO NOT EDIT)
import sys, os
sys.path.append("/home/shikim/pynta")
#sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))
from ase import Atom, Atoms
import copy
from ase.io import read, write, Trajectory
from acat.settings import CustomSurface
from acat.adsorption_sites import SlabAdsorptionSites
from ase.build import surface,fcc111
from ase.visualize import view
from site_analysis import *


In [ ]:

# Load pre-made slab
slab=read('/home/shikim/pynta/pynta/preprocessing/custom_surfaces/defects/pt111_defect_middle.xyz')

nslab = len(slab)
adsorbate_height = 2    
site_bond_cutoff = 1.5
view(slab,viewer='x3d')
pt111_defect = CustomSurface(slab, n_layers=4 )

In [ ]:
cas = SlabAdsorptionSites(slab, surface=pt111_defect, composition_effect=True)  # ACAT has surface, custom does not find them all!
single_geoms, single_sites_lists = generate_unique_sites(
        slab,
        cas.get_sites(),
        nslab,
        site_bond_cutoff,
        adsorbate_height
    )

In [ ]:

# Extract site graphs
admols, geom_indices = classify_all_sites(
    single_geoms, single_sites_lists
)


In [ ]:

# Graph isomorphism clustering
iso_mat, clusters = cluster_isomorphic_graphs(admols)


In [ ]:

# Pairwise strict isomorphism (PRINT)
print("Pairwise strict isomorphism:")
for i in range(len(admols)):
    for j in range(i + 1, len(admols)):
        print(f"{i} vs {j} =", iso_mat[i, j])


In [ ]:

# Assuming single_sites_lists and clusters are defined# Assuming single_sites_lists and clusters are defined
updated_sites, site_mapping = update_site_labels(single_sites_lists, clusters)

In [ ]:
write_trajectory_pynta(slab, cas, nslab, site_bond_cutoff, adsorbate_height)

In [ ]:
save_neighbor_site_list_to_json(cas)

# defects

In [ ]:

# Load pre-made slab
slab=read('/home/shikim/pynta/pynta/preprocessing/custom_surfaces/defects/pt111_defect_middle.xyz')
pt111_defect = CustomSurface(slab, n_layers=4 )
nslab = len(slab)
adsorbate_height = 1    
site_bond_cutoff = 1.5
view(slab,viewer='x3d')


In [24]:

# Load pre-made slab but no defect
slab=read('/home/shikim/pynta/pynta/preprocessing/custom_surfaces/defects/pt111_no_defect.xyz')
pt111_defect = CustomSurface(slab, n_layers=4 )
nslab = len(slab)
adsorbate_height = 1    
site_bond_cutoff = 1.5
view(slab,viewer='x3d')


In [25]:
defect_sites, _ = detect_vacancy_sites_from_coordination(slab, nslab)

In [26]:
print(f"Found {len(defect_sites)} defect site(s)")
for s in defect_sites:
    x, y, z = s["position"]
    print(f"{s.get('name','defect')}: x={x:.3f}  y={y:.3f}  z={z:.3f}  site={s.get('site')}")

Found 0 defect site(s)


In [27]:
pos = np.array([s["position"] for s in defect_sites])
print(pos)

[]


In [28]:
xyz_path = '/home/shikim/pynta/pynta/preprocessing/custom_surfaces/defects/pt111_no_defect.xyz'       # if geo already includes adsorbates, slab should be just slab atoms
geo = read(xyz_path)
sites = cas.get_sites()       # or however you build sites
nslab = len(slab)                 # your slab atom count
cas = SlabAdsorptionSites(slab, surface=pt111_defect, composition_effect=True)  

In [29]:
def my_get_sites(atoms):
    # if your CAS takes atoms as input, adapt accordingly:
    return cas.get_sites()  # <-- replace with cas.get_sites(atoms) if that's your API

In [30]:
cas.get_termination()
print(cas.get_termination())

([27, 28, 29, 30, 31, 32, 33, 34, 35], [18, 19, 20, 21, 22, 23, 24, 25, 26])


In [31]:
print(cas.get_unique_sites())
print(len(cas.get_unique_sites()))

[{'site': 'ontop', 'surface': 'CustomSurface', 'morphology': 'terrace', 'position': array([12.47320201,  7.20148072, 19.00418576]), 'normal': array([-3.534e-05,  8.600e-07,  1.000e+00]), 'indices': (27,), 'composition': 'Pt', 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([27])}, {'site': 'bridge', 'surface': 'CustomSurface', 'morphology': 'terrace', 'position': array([ 5.54376719,  7.20142653, 19.00491345]), 'normal': array([-1.2011000e-04,  3.0370000e-05,  9.9999999e-01]), 'indices': (27, 28), 'composition': 'PtPt', 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([27, 28])}, {'site': '3fold', 'surface': 'CustomSurface', 'morphology': 'terrace', 'position': array([ 1.38594787,  0.80021856, 19.0051173 ]), 'normal': array([-5.2499000e-04, -2.5474000e-04,  9.9999983e-01]), 'indices': (27, 28, 30), 'composition': 'PtPtPt', 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([27, 28, 30])}]
3


# Input to generate all sites including vacancy

In [33]:

# -------------------------
# inputs / objects
# -------------------------
xyz_path = "/home/shikim/pynta/pynta/preprocessing/custom_surfaces/defects/pt111_no_defect.xyz"

geo = read(xyz_path)

# slab should be slab-only atoms (no adsorbates). If geo is slab-only, this is fine:
slab = geo.copy()
nslab = len(slab)

# build CAS (ACAT)
cas = SlabAdsorptionSites(slab, surface=pt111_defect, composition_effect=True)

# the "sites" list used by generate_unique_site_additions_vacancy for normal (terrace) additions
sites = cas.get_sites()

# -------------------------
# wrapper: get_sites(atoms) for ANY geometry
# -------------------------
def my_get_sites(atoms):
    """
    ACAT's SlabAdsorptionSites is tied to the slab it was constructed with.
    So we rebuild it for the slab part of `atoms` and then call get_sites().
    """
    slab_part = atoms[:nslab].copy()
    cas_local = SlabAdsorptionSites(slab_part, surface=pt111_defect, composition_effect=True)
    return cas_local.get_sites()

# -------------------------
# run vacancy workflow
# -------------------------
(geom_all,
 params_all,
 sites_lists_all,
 maxcn_geoms,
 maxcn_meta,
 drop_geoms,
 drop_meta) = generate_unique_site_additions_vacancy(
    geo=geo,
    sites=sites,          # terrace sites from original slab CAS
    slab=slab,
    nslab=nslab,
    site_bond_cutoff=site_bond_cutoff,
    xyz_path=xyz_path,
    get_sites=my_get_sites,   # <-- key line

    tag_symbol="Ne",
    dz=0.1,
    stable_steps=3,
    save_all_drop_steps=True,
)

# -------------------------
# write outputs (optional)
# -------------------------
from ase.io import write
write("geom_all.traj", geom_all)
write("drop_steps.traj", drop_geoms)
write("maxcn_geoms.traj", maxcn_geoms)

# save sites_lists_all to JSON using your existing utility
save_sites_to_json(sites_lists_all, filename="geom_all_sites_lists.json")

Sites data saved to 'geom_all_sites_lists.json' with None replaced by 'null'.


In [ ]:
(geoms,
 params,
 slists,
 maxcn_geoms,
 maxcn_params,
 maxcn_sites,
 maxcn_meta,
 drop_geoms,
 drop_meta) = generate_unique_site_additions_vacancy(
    geo=geo,
    sites=sites,
    slab=slab,
    nslab=nslab,
    site_bond_cutoff=site_bond_cutoff,
    xyz_path=xyz_path,
    tag_symbol="Ne",   # choose noble gas
    get_sites=my_get_sites,   # <-- key line
    dz=0.1,
    max_drop=10.0,
    margin=0.25,
    stable_steps=3,    # patience (stop after 3 steps without cn increase)
    save_all_drop_steps=True,
)

from ase.io import write
write("vacancy_drop_steps.traj", drop_geoms)   # full drop trajectory
write("vacancy_max_cn.traj", maxcn_geoms)      # one frame per defect: max-cn position
write("vacancy_geoms.traj", geoms)
# Append the max-cn geometries onto the main geometry list
geoms_all = geoms + maxcn_geoms

from ase.io import write
write("vacancy_geoms_plus_maxcn.traj", geoms_all)

# quick inspection
for m in maxcn_meta:
    print("defect", m["defect_id"], "max_cn", m["cn"], "best_step", m["best_step"])


In [ ]:
#get site info 
from acat.adsorption_sites import SlabAdsorptionSites
atoms = read("vacancy_max_cn.xyz")
cas = SlabAdsorptionSites(atoms, surface=pt111_defect, composition_effect=True)  
site = cas.get_site() # Use cas.get_sites() to get all sites
print(site)

In [ ]:
geoms, site_bond_params_lists, sites_lists, *_ = generate_unique_site_additions_vacancy(
    geo=geo,
    sites=sites,
    slab=slab,
    nslab=nslab,
    site_bond_cutoff=site_bond_cutoff,
    xyz_path=xyz_path,
)

save_sites_to_json(sites_lists, filename="vacancy_sites_lists.json")

In [ ]:
unique_frames = write_unique_sites_with_drop_traj(
    drop_geoms, drop_meta,
    out_traj="vacancy_unique_sites_with_drop.traj",
    tol=0.25,
)
print("Wrote", len(unique_frames), "unique drop-site frames")